Runs the NER pipeline on several articles to evaluate the accuracy.

## Imports & setup

In [ ]:
# These commands auto-reload all imported modules, so that you don't have to rerun `helper_functions.py` if you make changes to it.

# load the autoreload extension
%load_ext autoreload 

# auto-reload all modules
%autoreload 2

In [ ]:
from flair.models import SequenceTagger
import pandas as pd
import ner

In [ ]:
# Initialize NER models

date_tagger = SequenceTagger.load("flair/ner-english-ontonotes-large") # 18-class Flair model that can extract dates and times
loc_tagger = SequenceTagger.load("Saisam/Inquirer_ner_loc") # initialize the model

## Load data

In [ ]:
df = pd.read_csv("police_reports.csv") # all texts
df.head()

We will filter the dataframe to only texts that are less than 3000 characters, since it takes too much computational time to run NER on long texts.

In [ ]:
small_df = df.loc[df["text"].str.len() < 3000].reset_index(drop=True)[:20].sort_values(by="text") # The dataframe to test NER on. Get only the first 20 texts
small_df

The first 20 rows have a variety of documents, such as articles, court cases, and other legal documents. So, we will see how NER performs across many different types of text.

## Test NER models

In [141]:
dates_df = ner.run_ner_on_dataframe(date_tagger, small_df, conf_thresh=None, entity_tag="DATE", splitting_method="every_n_words", n = 100) # tag all dates

In [142]:
dates_df

,text,text_id,entities,scores
0,Home Legislative File 2021-01132 RCA Legal s...,0,2021-10-20,0.999993
1,Home Legislative File 2021-01132 RCA Legal s...,0,2021-12-04,0.999784
2,Home Legislative File 2021-01132 RCA Legal s...,0,2021-12-04,0.999633
3,"Hennepin County 300 South Sixth Street, Minne...",1,2021-12-21,0.999994
4,4/12/2021 Officers ID'd in Brooklyn Center fat...,11,2019-09-06,0.999989
5,4/12/2021 Officers ID'd in Brooklyn Center fat...,11,2019-09-06,0.999992
6,4/12/2021 Officers ID'd in Brooklyn Center fat...,11,2019-09-06,0.999985
7,4/12/2021 Officers ID'd in Brooklyn Center fat...,11,2021-04-12,0.999988
8,4/12/2021 Officers ID'd in Brooklyn Center fat...,11,2021-04-12,0.999994
9,Home Legislative File 2022-00239 RCA Legal S...,14,2022-03-07,0.999994


In [143]:
# Detect all locations in the first 20 documents, with no threshold
locations_df = ner.run_ner_on_dataframe(loc_tagger, small_df, conf_thresh=None, entity_tag=None, splitting_method="by_sentences")

In [144]:
locations_df

,text,text_id,entities,scores
0,Home Legislative File 2021-01132 RCA Legal s...,0,City Aorney's Office,0.882122
1,Home Legislative File 2021-01132 RCA Legal s...,0,City of Minneapolis,0.924146
2,Home Legislative File 2021-01132 RCA Legal s...,0,Government,0.511303
3,Home Legislative File 2021-01132 RCA Legal s...,0,Neighborhood,0.571632
4,Home Legislative File 2021-01132 RCA Legal s...,0,Ward Neighborhood Address 1,0.754386
...,...,...,...,...
142,Home Legislative File 2022-00238 RCA Legal S...,63,Ward Neighborhood Address 1,0.918567
143,Too_LongOPCR Case #17-03832 Table of Contents...,65,MPD SIC b,0.78353
144,Too_LongOPCR Case #17-03832 Table of Contents...,65,Minneapolis Police,0.806397
145,Too_LongOPCR Case #17-03832 Table of Contents...,65,Minneapolis Police,0.862937


In [145]:
for id in small_df["text_id"]:
    print(f"\n{'~'*50} Text {id}: {'~'*50}")
    ner.print_text(small_df.loc[small_df["text_id"] == id, 'text'].values[0])

    print(f"\n{'-'*50} Dates: {'-'*50}")
    display(dates_df.loc[dates_df["text_id"] == id, ["entities", "scores"]])

    print(f"\n{'-'*50} Locations: {'-'*50}")

    # for this print statement, display an unlimited number of characters in a cell
    with pd.option_context('display.max_colwidth', None):
        display(locations_df.loc[locations_df["text_id"] == id, ["entities", "scores"]])


~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~ Text 43: ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
/ MPR News Notes on the news from the Twin Cities Minneapolis cops lauded for
arresting anti- Somali attacker Laura Yuen December 13, 2011, 5:22 PM Remember these guys? Minneapolis
police officers Abdiwahab Ali, left, and Mohamed Abdullahi were profiled in my Sept. 8 piece
on what it's like to be Muslim in Minnesota. Despite working on the front lines
of fighting crime, the two beat cops spoke of additional security measures they faced while
traveling through U.S. airports since the 2001 terrorist attacks. Today, the folks at the Department
of Homeland Security singled them out — to say thank you. The two men, along
with Somali community liaison Officer Jeanine Brudenell and crime-prevention specialist Ahmed Hassan, were honored today
with awards of appreciation for their work on a case resulting in a successful federal
hate-crime / About the blogger prosecution. In May 2010,

,entities,scores
23,2011-12-13,0.999988



-------------------------------------------------- Locations: --------------------------------------------------


,entities,scores
61,Africa,0.996952
62,Cedar-Riverside,0.993011
63,Department of Homeland Security,0.992475
64,Homeland Security,0.882198
65,Minneapolis,0.990084
66,Minnesota,0.997117
67,Somali,0.774299
68,Transportation Security Administration,0.852598
69,Twin Cities,0.993049
70,Twin Cities Minneapolis,0.974132



~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~ Text 48: ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
2/22/2021 BCA investigation complete in St. Paul officer-involved shooting of Billy Hughes https://www.fox9.com/news/bca-investigation-complete-in-st-paul-officer-involved-shooting-of-billy-hughes 1/2 BCA
investigation complete in St. Paul officer- involved shooting of Billy Hughes ST. PAUL, Minn. (KMSP)
- Investigators with the Minnesota Bureau of Criminal Apprehension have completed their investigation of the
deadly oÞcer- involved shooting of Billy Hughes and are turning over the case to the
Ramsey County Attorney's OÞce for review. On Aug 5., St. Paul police oÞcers responded to
a 911 call of shots Õred in Hughes' housing complex on the 900 block of
St. Anthony Avenue. Hughes, 45, was armed with a handgun when he was shot and
killed by St. Paul Police OÞcers Vincent Adams and Matthew Jones. A few weeks later,
the St. Paul Police Department released the body camera footage 

...

,entities,scores
72,City of Minneapolis,0.967181
73,UNITED STATES DISTRICT COURT District of Minnesota Darrell,0.927013



~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~ Text 65: ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
Too_LongOPCR Case #17-03832 Table of Contents 1. Table of Contents 2. Minneapolis Police Department Form
#3401 3. Investigative Summary 4. Statement of Sergeant Ali 5. Watch Commander's Log 6. MPD
Preliminary Incident Report 7. CAPRS MP 17-049029 8. Inform Browser (VISINET) 17-049029 a. VISINET AVL
Data 9. Minnesota State Accident Report 10. MPD Accident Review Findings 11. Crash Data Retrieval
(CDR ) Records 12. Route Map 13. Scene Photographs 14. Media Files a. Video from
MPD SIC b. MECC Audio Capture 15. Workforce Director Off-Duty Records 16. Minneapolis Police Department
Policy & Procedures 17. Employee Profile CUAPB002647

-------------------------------------------------- Dates: --------------------------------------------------


,entities,scores
41,NaN,NaN



-------------------------------------------------- Locations: --------------------------------------------------


,entities,scores
143,MPD SIC b,0.78353
144,Minneapolis Police,0.806397
145,Minneapolis Police,0.862937
146,Minnesota State,0.784632
